# OrigamiPipeline Tutorial: Training on Real Data

This notebook provides a step-by-step guide to using the `OrigamiPipeline` API for training, saving, and using ORIGAMI models on real-world data.

**What you'll learn:**
1. Loading and exploring JSON data
2. Training with default and custom configurations
3. Saving and loading trained pipelines
4. Making predictions on new data
5. Generating synthetic samples
6. Extracting embeddings for downstream tasks

**Dataset:** We'll use the UCI Car Evaluation dataset, which contains nested JSON objects describing car attributes and their acceptability ratings.

## 1. Setup and Imports

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch

from origami import OrigamiPipeline, PipelineConfig

# For reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.9.1


## 2. Load and Explore the Data

The car dataset contains nested JSON objects with car attributes and acceptability ratings.

In [2]:
# Load data from JSONL file
data_path = Path("../datasets/car.jsonl")

with open(data_path) as f:
    raw_data = [json.loads(line) for line in f]

print(f"Loaded {len(raw_data)} records")
print(f"\nSample record:")
print(json.dumps(raw_data[0], indent=2))

Loaded 1728 records

Sample record:
{
  "_id": {
    "$oid": "66c32b7fac3485a3c76f7e74"
  },
  "PRICE": {
    "buying": "vhigh",
    "maint": "vhigh"
  },
  "TECH": {
    "COMFORT": {
      "doors": 2,
      "persons": 2,
      "lug_boot": "small"
    },
    "safety": "low"
  },
  "target": "unacc"
}


In [3]:
# Remove MongoDB _id field (not useful for modeling)
data = [{k: v for k, v in doc.items() if k != "_id"} for doc in raw_data]

print("Cleaned record:")
print(json.dumps(data[0], indent=2))

Cleaned record:
{
  "PRICE": {
    "buying": "vhigh",
    "maint": "vhigh"
  },
  "TECH": {
    "COMFORT": {
      "doors": 2,
      "persons": 2,
      "lug_boot": "small"
    },
    "safety": "low"
  },
  "target": "unacc"
}


In [4]:
# Explore the data structure
print("Data structure:")
print("- PRICE: buying price and maintenance cost")
print("- TECH: technical specs (comfort features + safety)")
print("- target: acceptability rating\n")

# Count target values
target_counts = Counter(d["target"] for d in data)
print("Target distribution:")
for target, count in sorted(target_counts.items()):
    print(f"  {target}: {count} ({count / len(data) * 100:.1f}%)")

Data structure:
- PRICE: buying price and maintenance cost
- TECH: technical specs (comfort features + safety)
- target: acceptability rating

Target distribution:
  acc: 384 (22.2%)
  good: 69 (4.0%)
  unacc: 1210 (70.0%)
  vgood: 65 (3.8%)


In [5]:
# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

Train set: 1382 records
Eval set: 346 records


## 3. Training with Default Configuration

The simplest way to use `OrigamiPipeline` is with default settings. Just create a pipeline and call `fit()`.

In [6]:
from origami.training import TableLogCallback

callback = TableLogCallback(target_key="target")

In [7]:
# Create pipeline with default config
pipeline_default = OrigamiPipeline()

# View default configuration
print("Default configuration:")
print(pipeline_default)

Default configuration:
OrigamiPipeline(numeric_mode='none', not fitted)


In [9]:
# Train with defaults (quick training for demonstration)
pipeline_default.fit(train_data, eval_data=eval_data, epochs=50, callbacks=[callback], verbose=False)

print(f"\nTraining complete!")
print(f"Model parameters: {pipeline_default._model.get_num_parameters():,}")

| step: 10 | epoch: 0 | lr: 1.00e-05 | batch_dt: 18ms | train_loss: 2.6057 |
| step: 20 | epoch: 0 | lr: 2.00e-05 | batch_dt: 19ms | train_loss: 2.5017 |
| step: 30 | epoch: 0 | lr: 3.00e-05 | batch_dt: 18ms | train_loss: 2.3772 |
| step: 40 | epoch: 0 | lr: 4.00e-05 | batch_dt: 17ms | train_loss: 2.2250 |
| step: 50 | epoch: 1 | lr: 5.00e-05 | batch_dt: 18ms | train_loss: 2.0567 |
| step: 60 | epoch: 1 | lr: 6.00e-05 | batch_dt: 19ms | train_loss: 1.8858 |
| step: 70 | epoch: 1 | lr: 7.00e-05 | batch_dt: 18ms | train_loss: 1.7339 |
| step: 80 | epoch: 1 | lr: 8.00e-05 | batch_dt: 18ms | train_loss: 1.5696 |
| step: 90 | epoch: 2 | lr: 9.00e-05 | batch_dt: 18ms | train_loss: 1.3971 |
| step: 100 | epoch: 2 | lr: 1.00e-04 | batch_dt: 18ms | train_loss: 1.2548 | train_acc: 0.7100 | val_loss: 1.1671 | val_acc: 0.6600 |
| step: 110 | epoch: 2 | lr: 1.10e-04 | batch_dt: 18ms | train_loss: 1.1428 |
| step: 120 | epoch: 2 | lr: 1.20e-04 | batch_dt: 18ms | train_loss: 1.0474 |
| step: 130 | ep

## 4. Training with Custom Configuration

`PipelineConfig` lets you customize model architecture, training parameters, and preprocessing options.

In [ ]:
# Create a custom configuration
custom_config = PipelineConfig(
    # Model architecture
    d_model=128,  # Embedding dimension
    n_heads=4,  # Attention heads (must divide d_model)
    n_layers=4,  # Transformer layers
    d_ff=512,  # Feed-forward dimension
    dropout=0.1,  # Dropout rate
    # Position encoding
    kvpe_pooling="sum",  # How to combine path elements: "sum", "weighted", "gru"
    # Training parameters
    batch_size=32,
    learning_rate=1e-3,
    warmup_steps=100,
    shuffle_keys=True,  # Data augmentation via key order shuffling
    upscale_factor=2,  # Each sample seen 2x with different key orders
    # Numeric handling ("none", "discretize", or "scale")
    numeric_mode="none",  # Car dataset has no high-cardinality numerics
)

print("Custom configuration:")
print(f"  d_model: {custom_config.d_model}")
print(f"  n_layers: {custom_config.n_layers}")
print(f"  batch_size: {custom_config.batch_size}")
print(f"  upscale_factor: {custom_config.upscale_factor}")

In [ ]:
# Create and train pipeline with custom config
pipeline = OrigamiPipeline(custom_config)
pipeline.fit(train_data, eval_data=eval_data, epochs=20)

print(f"\nTraining complete!")
print(f"Model parameters: {pipeline._model.get_num_parameters():,}")

## 5. Saving and Loading Pipelines

Trained pipelines can be saved to disk and loaded later. The checkpoint includes the model weights, tokenizer vocabulary, configuration, and any preprocessors.

In [ ]:
# Save the trained pipeline
save_path = Path("car_pipeline.pt")
pipeline.save(save_path)

print(f"Pipeline saved to: {save_path}")
print(f"File size: {save_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Load the pipeline (demonstrates how to use a saved model)
save_path = Path("car_pipeline.pt")
pipeline = OrigamiPipeline.load(save_path)

print("Loaded pipeline:")
print(pipeline)

# Clean up
# save_path.unlink()
# print(f"\nCleaned up {save_path}")

## 6. Making Predictions

Use `predict()` to predict a missing field value given the rest of the object. This is useful for classification, imputation, and recommendation tasks.

In [ ]:
# Create a test object with missing target
test_obj = {
    "PRICE": {"buying": "low", "maint": "low"},
    "TECH": {"COMFORT": {"doors": 4, "persons": 4, "lug_boot": "big"}, "safety": "high"},
    "target": None,  # We want to predict this
}

print("Test object (target unknown):")
print(json.dumps(test_obj, indent=2))

In [ ]:
# Predict the target field
prediction = pipeline.predict(test_obj, target_key="target")
print(f"Predicted target: {prediction}")

# Get top-k predictions with probabilities using predict_proba
top_predictions = pipeline.predict_proba(test_obj, target_key="target", top_k=4)
print("\nTop predictions with probabilities:")
for value, prob in top_predictions:
    print(f"  {value}: {prob:.1%}")

In [ ]:
# Try different car configurations
test_cases = [
    # Expensive, unsafe car
    {
        "PRICE": {"buying": "vhigh", "maint": "vhigh"},
        "TECH": {"COMFORT": {"doors": 2, "persons": 2, "lug_boot": "small"}, "safety": "low"},
        "target": None,
    },
    # Cheap, safe family car
    {
        "PRICE": {"buying": "low", "maint": "low"},
        "TECH": {"COMFORT": {"doors": 4, "persons": "more", "lug_boot": "big"}, "safety": "high"},
        "target": None,
    },
    # Medium everything
    {
        "PRICE": {"buying": "med", "maint": "med"},
        "TECH": {"COMFORT": {"doors": 4, "persons": 4, "lug_boot": "med"}, "safety": "med"},
        "target": None,
    },
]

print("Predictions for different car configurations:")
print("=" * 60)
for i, test in enumerate(test_cases, 1):
    pred = pipeline.predict_proba(test, target_key="target", top_k=2)
    buying = test["PRICE"]["buying"]
    safety = test["TECH"]["safety"]
    print(f"\nCar {i}: buying={buying}, safety={safety}")
    print(f"  Predictions: {pred[0][0]} ({pred[0][1]:.0%}), {pred[1][0]} ({pred[1][1]:.0%})")

In [ ]:
# Batch prediction for efficiency
batch_results = pipeline.predict_batch(test_cases, target_key="target")

print("Batch predictions:")
for i, (result, _test) in enumerate(zip(batch_results, test_cases, strict=True), 1):
    print(f"  Car {i}: {result}")

## 7. Generating Synthetic Samples

Use `generate()` to sample new JSON objects from the learned distribution. This is useful for data augmentation, synthetic data generation, and understanding what the model has learned.

In [ ]:
# Generate synthetic car records
generated = pipeline.generate(num_samples=5, seed=42)

print("Generated car records:")
print("=" * 60)
for i, sample in enumerate(generated, 1):
    print(f"\nSample {i}:")
    print(json.dumps(sample, indent=2))

In [ ]:
# Generate with temperature control
# Lower temperature = more deterministic, higher = more diverse

print("Effect of temperature on generation:")
print("=" * 60)

for temp in [0.5, 1.0, 1.5]:
    samples = pipeline.generate(num_samples=3, temperature=temp, seed=123)
    targets = [s.get("target", "?") for s in samples]
    print(f"\nTemperature {temp}: targets = {targets}")

In [ ]:
# Generate many samples and analyze distribution
many_samples = pipeline.generate(num_samples=100, seed=42)

# Compare generated vs training distribution
gen_targets = Counter(s.get("target", "unknown") for s in many_samples)
train_targets = Counter(d["target"] for d in train_data)

print("Target distribution comparison:")
print(f"{'Target':<10} {'Training':>12} {'Generated':>12}")
print("-" * 36)
for target in sorted(set(train_targets.keys()) | set(gen_targets.keys())):
    train_pct = train_targets.get(target, 0) / len(train_data) * 100
    gen_pct = gen_targets.get(target, 0) / len(many_samples) * 100
    print(f"{target:<10} {train_pct:>11.1f}% {gen_pct:>11.1f}%")

## 8. Extracting Embeddings

Use `embed()` to get dense vector representations of JSON objects. These embeddings can be used for clustering, similarity search, visualization, or as features for downstream models.

In [ ]:
# Get embedding for a single object
sample_obj = train_data[0]
embedding = pipeline.embed(sample_obj)

print(f"Embedding shape: {embedding.shape}")
print(f"Embedding (first 10 dims): {embedding[:10].tolist()}")

In [ ]:
# Different pooling strategies
print("Pooling strategies:")
for pooling in ["mean", "max", "last"]:
    emb = pipeline.embed(sample_obj, pooling=pooling)
    print(f"  {pooling}: shape={emb.shape}, norm={np.linalg.norm(emb):.3f}")

In [ ]:
# Batch embedding for efficiency
batch_objects = train_data[:10]
embeddings = pipeline.embed_batch(batch_objects)

print(f"Batch embeddings shape: {embeddings.shape}")

In [ ]:
# Compute similarity between car records
from sklearn.metrics.pairwise import cosine_similarity

# Get embeddings for some eval samples
sample_cars = eval_data[:20]
car_embeddings = pipeline.embed_batch(sample_cars)

# Compute cosine similarity matrix
similarity = cosine_similarity(car_embeddings)

print(f"Similarity matrix shape: {similarity.shape}")
print("\nMost similar pairs (excluding self):")

# Find most similar pairs
similarity_no_diag = similarity.copy()
np.fill_diagonal(similarity_no_diag, -1)

for _ in range(3):
    max_idx = np.argmax(similarity_no_diag)
    i, j = max_idx // 20, max_idx % 20
    sim = similarity_no_diag[i, j]
    similarity_no_diag[i, j] = -1
    similarity_no_diag[j, i] = -1

    car_i_target = sample_cars[i]["target"]
    car_j_target = sample_cars[j]["target"]
    print(f"  Cars {i} ({car_i_target}) and {j} ({car_j_target}): similarity = {sim:.3f}")

In [ ]:
# Visualize embeddings with PCA
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Get embeddings for more samples
viz_samples = eval_data[:100]
viz_embeddings = pipeline.embed_batch(viz_samples)  # Already numpy array
viz_targets = [s["target"] for s in viz_samples]

# Reduce to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(viz_embeddings)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors = {"unacc": "red", "acc": "blue", "good": "green", "vgood": "purple"}
for target in colors:
    mask = [t == target for t in viz_targets]
    points = embeddings_2d[mask]
    if len(points) > 0:
        ax.scatter(points[:, 0], points[:, 1], c=colors[target], label=target, alpha=0.6)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("Car Embeddings by Target Class (PCA)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 9. Summary

This notebook demonstrated the key features of `OrigamiPipeline`:

### Configuration
```python
# Default config
pipeline = OrigamiPipeline()

# Custom config
config = PipelineConfig(
    d_model=128, n_layers=4, batch_size=32, ...
)
pipeline = OrigamiPipeline(config)
```

### Training
```python
pipeline.fit(train_data, eval_data=eval_data, epochs=10)
```

### Save/Load
```python
pipeline.save("model.pt")
pipeline = OrigamiPipeline.load("model.pt")
```

### Prediction
```python
# Single prediction (returns the value directly)
result = pipeline.predict(obj, target_key="field")

# Get probability distribution
probs = pipeline.predict_proba(obj, target_key="field")  # Returns dict

# Get top-k predictions with probabilities
top_k = pipeline.predict_proba(obj, target_key="field", top_k=5)  # Returns list of tuples

# Batch prediction (returns list of values)
results = pipeline.predict_batch(objects, target_key="field")
```

### Generation
```python
samples = pipeline.generate(num_samples=10, temperature=1.0, seed=42)
```

### Embeddings
```python
# Single embedding
emb = pipeline.embed(obj, pooling="mean")

# Batch embeddings
embs = pipeline.embed_batch(objects)
```

### Key Parameters
| Parameter | Description | Default |
|-----------|-------------|---------|
| `d_model` | Embedding dimension | 256 |
| `n_layers` | Transformer layers | 6 |
| `n_heads` | Attention heads | 8 |
| `numeric_mode` | "none", "discretize", "scale" | "none" |
| `shuffle_keys` | Key order augmentation | True |
| `upscale_factor` | Augmentation multiplier | 1 |